## Netherlands Rent Prediction

Given *data about properties in the Netherlands*, let's try to predict the **rent** for a given property.

We will use a random forest pipeline regression model to make our predictions.

Data source: https://www.kaggle.com/datasets/juangesino/netherlands-rent-properties

### Importing Libraries

In [2]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor

In [5]:
data = pd.read_json('archive/properties.json', lines=True)

In [6]:
data

,_id,externalId,areaRaw,areaSqm,city,coverImageUrl,crawlStatus,crawledAt,datesPublished,firstSeenAt,furnish,lastSeenAt,latitude,longitude,postalCode,postedAgo,propertyType,rawAvailability,rent,rentDetail,rentRaw,source,title,url,additionalCosts,additionalCostsRaw,deposit,depositRaw,descriptionNonTranslated,descriptionNonTranslatedRaw,descriptionTranslated,descriptionTranslatedRaw,detailsCrawledAt,energyLabel,gender,internet,isRoomActive,kitchen,living,matchAge,matchAgeBackup,matchCapacity,matchGender,matchGenderBackup,matchLanguages,matchStatus,matchStatusBackup,pageDescription,pageTitle,pets,registrationCost,registrationCostRaw,roommates,shower,smokingInside,toilet,userDisplayName,userId,userLastLoggedOn,userMemberSince,userPhotoUrl,additionalCostsDescription
0,{'$oid': '5d2b113a43cbfd7c77a998f4'},room-1686123,14 m2,14,Rotterdam,https://resources.kamernet.nl/image/913b4b03-5...,done,{'$date': '2019-07-26T22:18:23.018+0000'},"[{'$date': '2019-07-14T11:25:46.511+0000'}, {'...",{'$date': '2019-07-14T11:25:46.511+0000'},Unfurnished,{'$date': '2019-07-26T22:18:23.142+0000'},51.896601,4.514993,3074HN,4w,Room,26-06-'19 - Indefinite period,500,,"€ 500,-",kamernet,West-Varkenoordseweg,https://kamernet.nl/en/for-rent/room-rotterdam...,50.0,\n € 50\n ...,500.0,\n € 500\n ...,"Nice room for rent, accros the Feyenoord stadi...","\nNice room for rent, accros the Feyenoord sta...","Nice room for rent, accros the Feyenoord stadi...","\nNice room for rent, accros the Feyenoord sta...",{'$date': '2019-07-22T07:10:41.849+0000'},Unknown,Mixed,Yes,true,Shared,None,16 years -\n 99 years,16 years -\n 99 years,1 person,Not important,Not important,Not important,Not important,Not important,"Room for rent in Rotterdam, West-Varkenoordse...",Room for rent in Rotterdam €500 | Kamernet,No,0,\n € 0\n ...,5,Shared,No,Shared,Huize west,4680711.0,21-07-2019,26-06-2019,https://resources.kamernet.nl/Content/images/s...,NaN
1,{'$oid': '5d2b113a43cbfd7c77a9991a'},studio-1691193,30 m2,30,Amsterdam,https://resources.kamernet.nl/image/5e11d6b5-8...,done,{'$date': '2019-08-10T22:28:46.099+0000'},"[{'$date': '2019-07-14T11:25:46.677+0000'}, {'...",{'$date': '2019-07-14T11:25:46.677+0000'},Furnished,{'$date': '2019-08-10T22:28:46.229+0000'},52.370200,4.920721,1018AS,4w,Studio,15-08-'19 - Indefinite period,950,Utilities incl.,"€ 950,- Utilities incl.",kamernet,Parelstraat,https://kamernet.nl/en/for-rent/studio-amsterd...,0.0,\n € 0\n ...,895.0,\n € 895\n ...,"Efficiently furnished, with a large balcony, a...","\nEfficiently furnished, with a large balcony,...","Efficiently furnished, with a large balcony, a...","\nEfficiently furnished, with a large balcony,...",{'$date': '2019-07-22T06:29:33.112+0000'},Unknown,Unknown,Yes,true,Own,Own,18 years -\n 99 years,18 years -\n 99 years,1 person,Not important,Not important,Not important,"Working student, Working","Working student, Working","Studio for rent in Amsterdam, Parelstraat, fo...",Studio for rent in Amsterdam €950 | Kamernet,No,0,\n € 0\n ...,None,Own,No,Own,Cor,1865530.0,20-07-2019,05-01-2012,https://resources.kamernet.nl/Content/images/p...,NaN
2,{'$oid': '5d2b113a43cbfd7c77a99931'},room-1690545,11 m2,11,Amsterdam,https://resources.kamernet.nl/image/74b93a27-a...,done,{'$date': '2019-10-02T22:00:33.141+0000'},"[{'$date': '2019-07-14T11:25:46.834+0000'}, {'...",{'$date': '2019-07-14T11:25:46.834+0000'},Furnished,{'$date': '2019-10-02T22:00:33.264+0000'},52.350880,4.854786,1075SB,09 Jul,Room,01-08-'19 - Indefinite period,1000,Utilities incl.,"€ 1000,- Utilities incl.",kamernet,Zeilstraat,https://kamernet.nl/en/for-rent/room-amsterdam...,NaN,\n -\n ...,1000.0,\n € 1000\n ...,Kamer van 11m2 vlakbij het Vondelpark. Met een...,\nKamer van 11m2 vlakbij het Vondelpark. Met e...,Kamer van 11m2 vlakbij het Vondelpark. Met een...,\nKamer van 11m2 vlakbij het Vondelpark. Met e...,{'$date': '2019-07-21T08:44:32.816+0000'},Unknown,Mixed,Yes,true,Shared,Shared,16 years -\n 93 years,16 years -\n 93 years,1 person,Not important,Not i

In [7]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 46722 entries, 0 to 46721
Data columns (total 62 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   _id                          46722 non-null  object 
 1   externalId                   46722 non-null  str    
 2   areaRaw                      46722 non-null  str    
 3   areaSqm                      46722 non-null  int64  
 4   city                         46722 non-null  str    
 5   coverImageUrl                46722 non-null  str    
 6   crawlStatus                  46722 non-null  str    
 7   crawledAt                    46722 non-null  object 
 8   datesPublished               46722 non-null  object 
 9   firstSeenAt                  46722 non-null  object 
 10  furnish                      46722 non-null  str    
 11  lastSeenAt                   46722 non-null  object 
 12  latitude                     46722 non-null  float64
 13  longitude                  

### Preprocessing

In [19]:
df = data.copy()

In [20]:
# Use only select features
df = df[[
    'areaSqm',
    'city',
    'furnish',
    'latitude',
    'longitude',
    'propertyType',
    'rent',
    'internet',
    'kitchen',
    'living',
    'pets',
    'shower',
    'smokingInside',
    'toilet'
]]

df

,areaSqm,city,furnish,latitude,longitude,propertyType,rent,internet,kitchen,living,pets,shower,smokingInside,toilet
0,14,Rotterdam,Unfurnished,51.896601,4.514993,Room,500,Yes,Shared,None,No,Shared,No,Shared
1,30,Amsterdam,Furnished,52.370200,4.920721,Studio,950,Yes,Own,Own,No,Own,No,Own
2,11,Amsterdam,Furnished,52.350880,4.854786,Room,1000,Yes,Shared,Shared,Yes,Shared,Yes,Shared
3,16,Assen,Unfurnished,53.013494,6.561012,Room,290,Yes,Shared,None,No,Shared,Yes,Shared
4,22,Rotterdam,Unfurnished,51.932871,4.479732,Room,475,Unknown,Own,Own,No,Shared,No,Shared
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46717,28,Rotterdam,Furnished,51.928624,4.507187,Room,800,Yes,Shared,Shared,No,Shared,No,Shared
46718,16,Harmelen,Furnished,52.086568,4.959942,Room,400,Yes,Shared,Shared,No,Shared,No,Shared
46719,30,Rotterdam,Furnished,51.928624,4.507187,Room,950,Yes,Shared,Shared,No,Shared,No,Shared
46720,35,Rotterdam,Furnished,51.928624,4.507187,Room,1050,Yes,Shared,Shared,No,Shared,No,Shared


In [21]:
df.isna().sum()

areaSqm            0
city               0
furnish            0
latitude           0
longitude          0
propertyType       0
rent               0
internet         100
kitchen          100
living           100
pets             100
shower           100
smokingInside    100
toilet           100
dtype: int64

In [22]:
data['crawlStatus'].value_counts()

crawlStatus
done           46622
unavailable      100
Name: count, dtype: int64

In [23]:
data.query("crawlStatus == 'unavailable'")

,_id,externalId,areaRaw,areaSqm,city,coverImageUrl,crawlStatus,crawledAt,datesPublished,firstSeenAt,furnish,lastSeenAt,latitude,longitude,postalCode,postedAgo,propertyType,rawAvailability,rent,rentDetail,rentRaw,source,title,url,additionalCosts,additionalCostsRaw,deposit,depositRaw,descriptionNonTranslated,descriptionNonTranslatedRaw,descriptionTranslated,descriptionTranslatedRaw,detailsCrawledAt,energyLabel,gender,internet,isRoomActive,kitchen,living,matchAge,matchAgeBackup,matchCapacity,matchGender,matchGenderBackup,matchLanguages,matchStatus,matchStatusBackup,pageDescription,pageTitle,pets,registrationCost,registrationCostRaw,roommates,shower,smokingInside,toilet,userDisplayName,userId,userLastLoggedOn,userMemberSince,userPhotoUrl,additionalCostsDescription
77,{'$oid': '5d2b116743cbfd7c77a9a94b'},room-1691706,13 m2,13,Den Haag,https://resources.kamernet.nl/image/5b50a59e-c...,unavailable,{'$date': '2019-07-14T11:26:31.513+0000'},[{'$date': '2019-07-14T11:26:31.636+0000'}],{'$date': '2019-07-14T11:26:31.636+0000'},Furnished,{'$date': '2019-07-14T11:26:31.636+0000'},52.060570,4.300775,2525ZA,21h,Room,01-08-'19 - 01-09-'19,500,Utilities incl.,"€ 500,- Utilities incl.",kamernet,Jacob Schorerlaan,https://kamernet.nl/en/for-rent/room-den-haag/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'$date': '2019-07-22T07:13:33.837+0000'},NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,{'$oid': '5d2b116843cbfd7c77a9a9c8'},room-1691699,14 m2,14,Den Haag,https://resources.kamernet.nl/image/2fff4552-5...,unavailable,{'$date': '2019-07-16T06:02:01.690+0000'},"[{'$date': '2019-07-14T11:26:32.074+0000'}, {'...",{'$date': '2019-07-14T11:26:32.074+0000'},Furnished,{'$date': '2019-07-16T06:02:01.833+0000'},52.060570,4.300775,2525ZA,3d,Room,22-07-'19 - 27-12-'19,500,Utilities incl.,"€ 500,- Utilities incl.",kamernet,Jacob Schorerlaan,https://kamernet.nl/en/for-rent/room-den-haag/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'$date': '2019-07-21T08:53:47.875+0000'},NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,{'$oid': '5d2b117043cbfd7c77a9ac67'},room-1691664,20 m2,20,Ubbena,https://resources.kamernet.nl/image/6160f67d-5...,unavailable,{'$date': '2019-07-16T06:02:12.010+0000'},"[{'$date': '2019-07-14T11:26:40.809+0000'}, {'...",{'$date': '2019-07-14T11:26:40.809+0000'},Furnished,{'$date': '2019-07-16T06:02:12.132+0000'},53.054733,6.589102,9492TG,3d,Room,13-07-'19 - Indefinite period,20,Utilities incl.,"€ 20,- Utilities incl.",kamernet,Taarloseweg,https://kamernet.nl/en/for-rent/room-ubbena/ta...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'$date': '2019-07-22T07:10:25.755+0000'},NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
116,{'$oid': '5d2b117c43cbfd7c77a9b0dd'},room-1685795,18 m2,18,Groningen,https://resources.kamernet.nl/image/3bd7fa60-b...,unavailable,{'$date': '2019-07-15T00:01:48.442+0000'},"[{'$date': '2019-07-14T11:26:52.485+0000'}, {'...",{'$date': '2019-07-14T11:26:52.485+0000'},Uncarpeted,{'$date': '2019-07-15T00:01:48.564+0000'},53.232240,6.572627,9715AN,3w,Room,01-08-'19 - Indefinite period,355,Utilities incl.,"€ 355,- Utilities incl.",kamernet,Korreweg,https://kamernet.nl/en/for-rent/room-groningen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{'$date': '2019-07-21T09:08:27.206+0000'},NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
231,{'$oid': '5d2b11bf43cbfd7c77a9cbc1'},room-1691430,10 m2,10,Haarlem,https://resources.kamernet.nl/Content/images/p...,unavailable,{'$date': '2019-07-17T22:11:12.102+0000'},"[{'$date': '2019-07-14T11:27:59.797+0000'}, {'...",{'$date': '2019-07-14T11:27:59.797+0000'},Furnished,{'$date': '2019-07-17T22:11:12.223+0000'},52.365043,4.659657,2035VE,5d,Room,12-07-'19 - Indefinite period,350,Utilities incl.,"€ 350,- Utilities incl.",kamernet,Ekamastraat,https://kamernet.nl/en/f

In [24]:
# Drop missing rows
missing_rows = data.query("crawlStatus == 'unavailable'").index
df = df.drop(missing_rows, axis=0).reset_index(drop=True)

In [25]:
df.isna().sum()

areaSqm          0
city             0
furnish          0
latitude         0
longitude        0
propertyType     0
rent             0
internet         0
kitchen          0
living           0
pets             0
shower           0
smokingInside    0
toilet           0
dtype: int64

In [26]:
df

,areaSqm,city,furnish,latitude,longitude,propertyType,rent,internet,kitchen,living,pets,shower,smokingInside,toilet
0,14,Rotterdam,Unfurnished,51.896601,4.514993,Room,500,Yes,Shared,None,No,Shared,No,Shared
1,30,Amsterdam,Furnished,52.370200,4.920721,Studio,950,Yes,Own,Own,No,Own,No,Own
2,11,Amsterdam,Furnished,52.350880,4.854786,Room,1000,Yes,Shared,Shared,Yes,Shared,Yes,Shared
3,16,Assen,Unfurnished,53.013494,6.561012,Room,290,Yes,Shared,None,No,Shared,Yes,Shared
4,22,Rotterdam,Unfurnished,51.932871,4.479732,Room,475,Unknown,Own,Own,No,Shared,No,Shared
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46617,28,Rotterdam,Furnished,51.928624,4.507187,Room,800,Yes,Shared,Shared,No,Shared,No,Shared
46618,16,Harmelen,Furnished,52.086568,4.959942,Room,400,Yes,Shared,Shared,No,Shared,No,Shared
46619,30,Rotterdam,Furnished,51.928624,4.507187,Room,950,Yes,Shared,Shared,No,Shared,No,Shared
46620,35,Rotterdam,Furnished,51.928624,4.507187,Room,1050,Yes,Shared,Shared,No,Shared,No,Shared


In [27]:
df.describe()

,areaSqm,latitude,longitude,rent
count,46622.000000,46622.000000,46622.000000,46622.000000
mean,31.636974,52.201675,5.315371,667.974111
std,29.877284,0.517286,0.798836,416.830406
min,6.000000,50.770041,3.410016,1.000000
25%,14.000000,51.925374,4.712789,395.000000
50%,20.000000,52.162522,5.083407,550.000000
75%,40.000000,52.370420,5.896593,800.000000
max,675.000000,53.434608,7.206637,5999.000000


In [28]:
{column: list(df[column].unique()) for column in df.select_dtypes('str').columns}

{'city': ['Rotterdam',
  'Amsterdam',
  'Assen',
  'Groningen',
  'Zeist',
  'Maastricht',
  'Callantsoog',
  'Alphen aan den Rijn',
  'Tilburg',
  'Enschede',
  'Leeuwarden',
  'Eindhoven',
  'Wageningen',
  'Diemen',
  'Utrecht',
  'Almere',
  'Alkmaar',
  'Harderwijk',
  'Hilversum',
  'Delft',
  'Den Bosch',
  'Stoutenburg',
  'Leiden',
  'Den Haag',
  'Boxtel',
  'Badhoevedorp',
  'Veenendaal',
  'Amstelveen',
  'Nijmegen',
  'Venlo',
  'Zwolle',
  'Ubbena',
  'Arnhem',
  'Leimuiden',
  'Riel',
  'Nieuwegein',
  'Haren Gn',
  'Uitgeest',
  'Beverwijk',
  'Ede',
  'Nijkerk',
  'Amersfoort',
  'Loosdrecht',
  'Apeldoorn',
  'Vaals',
  'Velp',
  'Vlaardingen',
  'Montfoort',
  'Heemstede',
  'Breda',
  'Purmerend',
  'Baarn',
  'Spijkenisse',
  'Deventer',
  'Hengelo',
  'Capelle aan den IJssel',
  'Bovenkarspel',
  'Weesp',
  'Harskamp',
  'Zeeland',
  'Waalre',
  'IJsselstein',
  'Pijnacker',
  'Sittard',
  'Putten',
  'Vlissingen',
  'Haarlem',
  'Rijswijk',
  'Zandvoort',
  'Zutp

In [29]:
# Encode improper values
df = df.replace({'': np.nan, 'Unknown': np.nan})

In [30]:
{column: list(df[column].unique()) for column in df.select_dtypes('str').columns}

{'city': ['Rotterdam',
  'Amsterdam',
  'Assen',
  'Groningen',
  'Zeist',
  'Maastricht',
  'Callantsoog',
  'Alphen aan den Rijn',
  'Tilburg',
  'Enschede',
  'Leeuwarden',
  'Eindhoven',
  'Wageningen',
  'Diemen',
  'Utrecht',
  'Almere',
  'Alkmaar',
  'Harderwijk',
  'Hilversum',
  'Delft',
  'Den Bosch',
  'Stoutenburg',
  'Leiden',
  'Den Haag',
  'Boxtel',
  'Badhoevedorp',
  'Veenendaal',
  'Amstelveen',
  'Nijmegen',
  'Venlo',
  'Zwolle',
  'Ubbena',
  'Arnhem',
  'Leimuiden',
  'Riel',
  'Nieuwegein',
  'Haren Gn',
  'Uitgeest',
  'Beverwijk',
  'Ede',
  'Nijkerk',
  'Amersfoort',
  'Loosdrecht',
  'Apeldoorn',
  'Vaals',
  'Velp',
  'Vlaardingen',
  'Montfoort',
  'Heemstede',
  'Breda',
  'Purmerend',
  'Baarn',
  'Spijkenisse',
  'Deventer',
  'Hengelo',
  'Capelle aan den IJssel',
  'Bovenkarspel',
  'Weesp',
  'Harskamp',
  'Zeeland',
  'Waalre',
  'IJsselstein',
  'Pijnacker',
  'Sittard',
  'Putten',
  'Vlissingen',
  'Haarlem',
  'Rijswijk',
  'Zandvoort',
  'Zutp

In [32]:
df.isna().mean()

areaSqm          0.000000
city             0.000000
furnish          0.007765
latitude         0.000000
longitude        0.000000
propertyType     0.000000
rent             0.000000
internet         0.174724
kitchen          0.163614
living           0.183411
pets             0.000000
shower           0.163678
smokingInside    0.000000
toilet           0.164300
dtype: float64

In [33]:
df['living'].mode()

0    Shared
Name: living, dtype: str

In [37]:
# Fill missing values
missing_value_columns = df.columns[df.isna().sum() > 0].tolist()

In [38]:
for column in missing_value_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

In [39]:
df.isna().sum()

areaSqm          0
city             0
furnish          0
latitude         0
longitude        0
propertyType     0
rent             0
internet         0
kitchen          0
living           0
pets             0
shower           0
smokingInside    0
toilet           0
dtype: int64

In [40]:
{column: list(df[column].unique()) for column in df.select_dtypes('str').columns}

{'city': ['Rotterdam',
  'Amsterdam',
  'Assen',
  'Groningen',
  'Zeist',
  'Maastricht',
  'Callantsoog',
  'Alphen aan den Rijn',
  'Tilburg',
  'Enschede',
  'Leeuwarden',
  'Eindhoven',
  'Wageningen',
  'Diemen',
  'Utrecht',
  'Almere',
  'Alkmaar',
  'Harderwijk',
  'Hilversum',
  'Delft',
  'Den Bosch',
  'Stoutenburg',
  'Leiden',
  'Den Haag',
  'Boxtel',
  'Badhoevedorp',
  'Veenendaal',
  'Amstelveen',
  'Nijmegen',
  'Venlo',
  'Zwolle',
  'Ubbena',
  'Arnhem',
  'Leimuiden',
  'Riel',
  'Nieuwegein',
  'Haren Gn',
  'Uitgeest',
  'Beverwijk',
  'Ede',
  'Nijkerk',
  'Amersfoort',
  'Loosdrecht',
  'Apeldoorn',
  'Vaals',
  'Velp',
  'Vlaardingen',
  'Montfoort',
  'Heemstede',
  'Breda',
  'Purmerend',
  'Baarn',
  'Spijkenisse',
  'Deventer',
  'Hengelo',
  'Capelle aan den IJssel',
  'Bovenkarspel',
  'Weesp',
  'Harskamp',
  'Zeeland',
  'Waalre',
  'IJsselstein',
  'Pijnacker',
  'Sittard',
  'Putten',
  'Vlissingen',
  'Haarlem',
  'Rijswijk',
  'Zandvoort',
  'Zutp

In [45]:
{column: len(df[column].unique()) for column in df.select_dtypes('str').columns}

{'city': 737,
 'furnish': 3,
 'propertyType': 5,
 'internet': 2,
 'kitchen': 3,
 'living': 3,
 'pets': 3,
 'shower': 3,
 'smokingInside': 3,
 'toilet': 3}

In [41]:
# Split df into X and y
y = df['rent']
X = df.drop('rent', axis=1)

In [42]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

### Building Pipeline and Training

In [43]:
X_train

,areaSqm,city,furnish,latitude,longitude,propertyType,internet,kitchen,living,pets,shower,smokingInside,toilet
29791,61,Rotterdam,Unfurnished,51.925125,4.486212,Apartment,Yes,Shared,Shared,No,Shared,No,Shared
44827,45,Rotterdam,Unfurnished,51.893369,4.517075,Apartment,Yes,Own,Own,No,Own,No,Own
37089,105,Amsterdam,Furnished,52.376979,4.839116,Apartment,Yes,Own,Own,No,Own,No,Own
13269,20,Delft,Uncarpeted,51.996010,4.352954,Room,Yes,Shared,Shared,By mutual agreement,Shared,Not important,Shared
14654,44,Rotterdam,Furnished,51.891709,4.480317,Apartment,Yes,Shared,Shared,No,Shared,No,Shared
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43723,11,Enschede,Unfurnished,52.234413,6.849043,Room,Yes,Shared,Shared,No,Shared,No,Shared
32511,21,Utrecht,Furnished,52.086425,5.125311,Room,Yes,Shared,Shared,No,Shared,No,Shared
5192,16,Groningen,Furnished,53.229623,6.524759,Room,Yes,Shared,Shared,No,Shared,No,Shared
12172,7,Utrecht,Furnished,52.102119,5.096010,Room,Yes,Shared,None,No,Shared,No,Shared


In [52]:
nominal_features = [
    'city',
    'furnish',
    'propertyType',
    'kitchen',
    'living',
    'pets',
    'shower',
    'smokingInside',
    'toilet'
]

binary_transformer = Pipeline(steps=[
      ('ordinal', OrdinalEncoder())
])

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('binary', binary_transformer, ['internet']),
    ('nominal', nominal_transformer, nominal_features)
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())  
])

In [53]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['areaSqm','city','furnish',...,'shower','smokingInside','toilet']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('binary', ...), ('nominal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of c

### Results

In [55]:
y_pred = model.predict(X_test)

In [56]:
y_pred

array([ 436.17      ,  963.74833333,  342.785     , ..., 1282.75      ,
        360.39      ,  428.89      ], shape=(13987,))

In [57]:
y_test

31410     400
34454     775
42517     345
36306     500
43045     800
         ... 
42138    1600
36266     795
7926     1250
25058     365
15087     475
Name: rent, Length: 13987, dtype: int64

In [58]:
y_test - y_pred

31410    -36.170000
34454   -188.748333
42517      2.215000
36306    -35.430000
43045    -53.366667
            ...    
42138      3.550000
36266     23.145000
7926     -32.750000
25058      4.610000
15087     46.110000
Name: rent, Length: 13987, dtype: float64

In [62]:
rmse = np.sqrt(np.mean((y_test - y_pred)**2))
rmse

np.float64(156.63801839446992)

In [63]:
y_test.describe()

count    13987.000000
mean       664.911561
std        413.872062
min          1.000000
25%        390.000000
50%        550.000000
75%        800.000000
max       5000.000000
Name: rent, dtype: float64

In [73]:
r2_score = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2))
r2_score

np.float64(0.8567504754894875)

In [79]:
print("RMSE: {:.2f}".format(rmse) )
print("R2 Score: {:.5f}".format(r2_score))

RMSE: 156.64
R2 Score: 0.85675
